# Gemini Robotics 좌표 Grounding 실습

이 노트북은 API 없이 `[y, x]` 0~1000 좌표를 픽셀과 테이블 좌표로 바꾸는 과정을 다룹니다. 모델 출력은 행동이 아니라 **검증 전 제안**이라는 원칙을 유지합니다.

## 1. 학습용 패키지 불러오기

`labs` 루트 또는 `notebooks` 폴더에서 실행해도 `src`를 찾도록 경로를 계산합니다.

In [ ]:
# Path는 현재 notebook 위치를 운영체제에 독립적으로 다룹니다.
from pathlib import Path
# sys.path에는 import할 package의 부모 폴더를 추가합니다.
import sys

# 현재 폴더가 notebooks라면 부모가 labs이고, 아니면 현재 폴더를 labs로 봅니다.
current = Path.cwd().resolve()
# src 폴더의 실제 위치를 선택합니다.
src = (current.parent / 'src') if current.name == 'notebooks' else (current / 'src')
# 중복 경로를 만들지 않고 문자열 경로를 import 검색 목록 앞에 넣습니다.
if str(src) not in sys.path:
    sys.path.insert(0, str(src))

# 좌표와 calibration 구현을 재사용합니다.
from gemini_robotics_learning.geometry import PlanarCalibration
# 생성형 문자열을 안전한 point로 바꾸는 parser입니다.
from gemini_robotics_learning.schemas import parse_point_detections

## 2. Gemini 형식의 응답 검증

공식 순서는 `[y, x]`입니다. 두 축을 구분하기 위해 서로 다른 값을 사용합니다.

In [ ]:
# 실제 모델처럼 설명과 JSON이 섞인 응답을 준비합니다.
response = 'result: [{"point":[250,750],"label":"blue block"}]'
# eval이 아니라 JSON parser와 좌표 범위 검사를 사용합니다.
detections = parse_point_detections(response, maximum_items=1)
# 첫 번째 검증 결과를 선택합니다.
target = detections[0]
# y=250, x=750인지 확인합니다.
target

## 3. Pixel 변환과 시각화

정규화 값 1000은 마지막 픽셀 index인 `width-1`, `height-1`에 대응합니다.

In [ ]:
# 예제 이미지 크기를 정합니다.
width, height = 640, 480
# [y,x]를 pixel 객체의 (x,y)로 바꿉니다.
pixel = target.point.to_pixel(width=width, height=height)
# 변환 결과를 출력합니다.
pixel

In [ ]:
# Matplotlib은 좌표 축 방향을 눈으로 확인하는 데 사용합니다.
import matplotlib.pyplot as plt
# 빈 카메라 frame과 같은 크기의 figure를 만듭니다.
fig, ax = plt.subplots(figsize=(8, 6))
# 이미지 좌표와 같도록 x 범위를 0~width로 설정합니다.
ax.set_xlim(0, width)
# 이미지 y는 아래로 증가하므로 height~0 순서로 뒤집습니다.
ax.set_ylim(height, 0)
# 검증된 point만 빨간 점으로 그립니다.
ax.scatter([pixel.x], [pixel.y], color='red', s=100)
# label을 point 옆에 표시합니다.
ax.text(pixel.x + 8, pixel.y, target.label)
# 축의 단위를 명시합니다.
ax.set_xlabel('pixel x')
ax.set_ylabel('pixel y')
# 격자로 축 뒤집힘을 쉽게 확인합니다.
ax.grid(True)
plt.show()

## 4. 평면 Homography

아래 matrix는 수식 연습용입니다. 실제 로봇에서는 calibration target으로 구한 matrix와 version을 사용해야 합니다.

In [ ]:
# pixel 1개를 0.5mm로 환산하고 원점을 이동하는 예시 matrix입니다.
calibration = PlanarCalibration(
    matrix=((0.0005, 0.0, -0.16),
            (0.0, 0.0005, -0.12),
            (0.0, 0.0, 1.0)),
    frame_id='table',
    units='meter',
)
# Pixel point를 table frame의 meter 좌표로 변환합니다.
world_xy = calibration.pixel_to_world(pixel)
# 좌표만 출력하며 실제 행동은 수행하지 않습니다.
print({'frame_id': calibration.frame_id, 'units': calibration.units, 'xy': world_xy})
print('ACTION DISABLED: safety validation and operator confirmation are required.')

## 연습 문제

1. `[100, 900]`을 넣고 point가 오른쪽 위에 그려지는지 확인하세요.
2. 1001 또는 -1 좌표가 거부되는지 확인하세요.
3. 같은 물체를 5회 질의했다고 가정한 point의 median과 spread를 계산하세요.
4. Calibration matrix를 바꿀 때 frame과 version을 함께 기록할 구조를 설계하세요.
5. 2D table 가정을 벗어난 물체에는 왜 depth가 필요한지 설명하세요.